# Phase 4 · Step 1 — Prompt v1 (baseline RAG prompt)

Runs the full Phase 2 pipeline (hybrid search + re-rank) to get real
chunks, then feeds them into the naive `answer_v1` prompt from
`backend/src/prompts/v1.py`. See
`learning/06_prompts/01_prompt_v1.md` for the why behind this design.

## Reload, re-chunk, re-embed, upsert

Same setup as notebook 06 — rebuilds the hybrid-search collection from
scratch so this notebook is self-contained.

In [1]:
import logging

logging.getLogger("numexpr").setLevel(logging.WARNING)

from pathlib import Path

from generate_sample_pdf import generate
from llama_index.core.node_parser import SentenceSplitter
from llama_index.readers.file import PDFReader
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, Modifier, SparseVectorParams, VectorParams

COLLECTION_NAME = "papermind_chunks"
STORAGE_PATH = Path("data/qdrant")
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SPARSE_MODEL = "Qdrant/bm25"
RERANK_MODEL = "Xenova/ms-marco-MiniLM-L-6-v2"

pdf_path = generate()
documents = PDFReader().load_data(file=pdf_path)
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

client = QdrantClient(path=str(STORAGE_PATH))
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={"dense": VectorParams(size=384, distance=Distance.COSINE)},
    sparse_vectors_config={"bm25": SparseVectorParams(modifier=Modifier.IDF)},
)

print(f"Produced {len(nodes)} chunks from {len(documents)} pages")

Produced 4 chunks from 4 pages


In [2]:
from uuid import NAMESPACE_URL, uuid5

from fastembed import SparseTextEmbedding, TextEmbedding
from qdrant_client.models import PointStruct, SparseVector

texts = [node.get_content() for node in nodes]

dense_embedder = TextEmbedding(model_name=DENSE_MODEL)
sparse_embedder = SparseTextEmbedding(model_name=SPARSE_MODEL)

dense_vectors = list(dense_embedder.embed(texts))
sparse_vectors = list(sparse_embedder.embed(texts))

page_counters: dict[str, int] = {}
points = []
for node, dense_vector, sparse_vector in zip(nodes, dense_vectors, sparse_vectors):
    page_label = node.metadata["page_label"]
    file_name = node.metadata["file_name"]
    chunk_index = page_counters.get(page_label, 0)
    page_counters[page_label] = chunk_index + 1

    point_id = str(uuid5(NAMESPACE_URL, f"{file_name}:{page_label}:{chunk_index}"))
    points.append(
        PointStruct(
            id=point_id,
            vector={
                "dense": dense_vector.tolist(),
                "bm25": SparseVector(
                    indices=sparse_vector.indices.tolist(),
                    values=sparse_vector.values.tolist(),
                ),
            },
            payload={
                "text": node.get_content(),
                "page_label": page_label,
                "file_name": file_name,
                "chunk_index": chunk_index,
            },
        )
    )

client.upsert(collection_name=COLLECTION_NAME, points=points)
print("Points count:", client.get_collection(COLLECTION_NAME).points_count)

Points count: 4


## Hybrid search + re-rank

Same as notebook 06: over-fetch 4 candidates with hybrid search, re-rank
with the cross-encoder, keep the top 2 as the final retrieved chunks.

In [3]:
import logging
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

from fastembed.rerank.cross_encoder import TextCrossEncoder
from qdrant_client.models import FusionQuery, Fusion, Prefetch

question = "How is citation accuracy measured in this paper?"

query_dense = next(dense_embedder.embed([question])).tolist()
query_sparse = next(sparse_embedder.embed([question]))

hybrid_response = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        Prefetch(query=query_dense, using="dense", limit=10),
        Prefetch(
            query=SparseVector(
                indices=query_sparse.indices.tolist(),
                values=query_sparse.values.tolist(),
            ),
            using="bm25",
            limit=10,
        ),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=4,
    with_payload=True,
)
candidates = hybrid_response.points

reranker = TextCrossEncoder(model_name=RERANK_MODEL)
candidate_texts = [c.payload["text"] for c in candidates]
rerank_scores = list(reranker.rerank(question, candidate_texts))
scored = sorted(zip(candidates, rerank_scores), key=lambda pair: pair[1], reverse=True)

TOP_K = 2
final_candidates = [c for c, _ in scored[:TOP_K]]
chunks = [c.payload for c in final_candidates]

for chunk in chunks:
    print(f"page {chunk['page_label']}: {chunk['text'][:60]}...")

page 1: Abstract
This paper studies retrieval-augmented generation f...
page 4: 3. Results
Across a held-out set of research papers, grounde...


## Answer with prompt v1

Needs `backend/.env` with `OPENAI_API_KEY` (set up in Phase 3). Real,
billed API call.

In [4]:
import logging
import sys

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

sys.path.insert(0, str(Path.cwd().parent))

from src.llm import get_llm
from src.prompts.v1 import answer, build_prompt

llm = get_llm("openai")

print("--- Prompt sent to the LLM ---")
print(build_prompt(question, chunks))
print()
print("--- Answer ---")
print(answer(question, chunks, llm))

--- Prompt sent to the LLM ---
Context:
Abstract
This paper studies retrieval-augmented generation for question answering over research
papers. We show that combining dense retrieval with a re-ranking stage improves citation
accuracy compared to retrieval alone.

3. Results
Across a held-out set of research papers, grounded answers with page citations were rated
as more trustworthy than ungrounded answers by human reviewers, even when the two answers
conveyed the same information.

Question: How is citation accuracy measured in this paper?

Answer the question using the context above.

--- Answer ---


2026-11-25 12:52:22,374 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The context does not provide specific details on how citation accuracy is measured in the paper. However, it indicates that the improvement in citation accuracy is assessed by comparing the performance of retrieval-augmented generation methods against retrieval alone, and it mentions that human reviewers rated grounded answers with page citations as more trustworthy than ungrounded answers. This suggests that citation accuracy may be indirectly measured through the perceived trustworthiness of answers that include citations.


## Manual check

No automated assertion here on purpose — v1 was never told to cite pages,
so we're reading the answer above and noting, honestly, whether it
mentions *which* page/section the information came from, or just answers
in prose with no source trail. That observation goes into the learning
doc's "What we did" and is exactly what v2 is meant to fix.

## Next

Phase 4 · Step 2 adds citation/grounding instructions to the prompt
(v2), tested against the same question + chunks for a fair comparison.